# NB04: Update ModelSEEDDatabase with New deltaG Values

Write the OPAM2-derived deltaG values (computed in NB03) into the
ModelSEEDDatabase reaction TSV files.

Columns: `deltag` (col 14, kJ/mol) and `deltagerr` (col 15, kJ/mol).

dGPredictor outputs are already in kJ/mol (same units as ModelSEEDDatabase).

In [1]:
import sys
import json
import glob
from pathlib import Path

import pandas as pd
import numpy as np
from tqdm import tqdm

MSDB_ROOT = Path('/tmp/ModelSEEDDatabase')
PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'

## 1. Load OPAM2-derived deltaG predictions

In [2]:
with open(DATA_DIR / 'dg_predictions_opam2.json') as f:
    new_results = json.load(f)

print(f'Loaded {len(new_results):,} reaction deltaG predictions')
print(f'Sample: {list(new_results.items())[:2]}')

Loaded 31,924 reaction deltaG predictions
Sample: [('rxn00001', {'dG_mean': -16.479271435241415, 'dG_std': 3.628592906553887, 'dG_model_only': 23.455338865109752, 'ddG0_pH_correction': -39.93461030035117}), ('rxn00002', {'dG_mean': -84.58516460195113, 'dG_std': 64.40527733061968, 'dG_model_only': -207.32960652083239, 'ddG0_pH_correction': 122.74444191888125})]


## 2. Update reaction TSV files

In [3]:
rxn_files = sorted(glob.glob(str(MSDB_ROOT / 'Biochemistry' / 'reaction_*.tsv')))

total_updated = 0
total_unchanged = 0
total_no_prediction = 0
old_values = []

for rxn_file in tqdm(rxn_files, desc='Updating reaction files'):
    df = pd.read_csv(rxn_file, sep='\t')
    file_updated = 0

    for idx, row in df.iterrows():
        rxn_id = row['id']
        if rxn_id not in new_results:
            total_no_prediction += 1
            continue

        pred = new_results[rxn_id]
        new_dg = round(pred['dG_mean'], 2)
        new_err = round(pred['dG_std'], 2)

        old_dg = row.get('deltag', '')
        old_err = row.get('deltagerr', '')

        old_values.append({
            'rxn_id': rxn_id,
            'old_deltag': old_dg,
            'old_deltagerr': old_err,
            'new_deltag': new_dg,
            'new_deltagerr': new_err,
        })

        if pd.notna(old_dg) and abs(float(old_dg) - new_dg) < 0.01:
            total_unchanged += 1
            continue

        df.at[idx, 'deltag'] = new_dg
        df.at[idx, 'deltagerr'] = new_err
        file_updated += 1
        total_updated += 1

    if file_updated > 0:
        df.to_csv(rxn_file, sep='\t', index=False)

print(f'\nTotal reactions updated: {total_updated:,}')
print(f'Total reactions unchanged: {total_unchanged:,}')
print(f'Total reactions with no prediction: {total_no_prediction:,}')

Updating reaction files:   0%|                                                                                       | 0/61 [00:00<?, ?it/s]

Updating reaction files:   3%|██▌                                                                            | 2/61 [00:00<00:03, 16.66it/s]

Updating reaction files:   7%|█████▏                                                                         | 4/61 [00:00<00:03, 17.19it/s]

Updating reaction files:  11%|█████████                                                                      | 7/61 [00:00<00:02, 19.63it/s]

Updating reaction files:  16%|████████████▊                                                                 | 10/61 [00:00<00:02, 21.88it/s]

...

Updating reaction files:  90%|██████████████████████████████████████████████████████████████████████▎       | 55/61 [00:02<00:00, 22.13it/s]

Updating reaction files:  95%|██████████████████████████████████████████████████████████████████████████▏   | 58/61 [00:02<00:00, 22.40it/s]

Updating reaction files: 100%|██████████████████████████████████████████████████████████████████████████████| 61/61 [00:02<00:00, 23.78it/s]

Updating reaction files: 100%|██████████████████████████████████████████████████████████████████████████████| 61/61 [00:02<00:00, 22.37it/s]


Total reactions updated: 28,957
Total reactions unchanged: 2,967
Total reactions with no prediction: 24,088


## 3. Save update log

In [4]:
df_log = pd.DataFrame(old_values)
df_log.to_csv(DATA_DIR / 'dg_update_log.tsv', sep='\t', index=False)
print(f'Saved {len(df_log):,} reaction update records to data/dg_update_log.tsv')

Saved 31,924 reaction update records to data/dg_update_log.tsv


## 4. Regenerate JSON from updated TSVs

In [5]:
import subprocess

reprint_script = MSDB_ROOT / 'Scripts' / 'Biochemistry' / 'Reprint_Biochemistry.py'
if reprint_script.exists():
    result = subprocess.run(
        [sys.executable, str(reprint_script)],
        cwd=str(MSDB_ROOT),
        capture_output=True, text=True, timeout=300
    )
    print('STDOUT:', result.stdout[-500:] if result.stdout else '(empty)')
    if result.returncode != 0:
        print('STDERR:', result.stderr[-500:])
    else:
        print('JSON regeneration complete.')
else:
    print(f'Reprint script not found at {reprint_script}')

STDOUT: (empty)
STDERR: Traceback (most recent call last):
  File "/tmp/ModelSEEDDatabase/Scripts/Biochemistry/Reprint_Biochemistry.py", line 4, in <module>
    from BiochemPy import Reactions, Compounds
ModuleNotFoundError: No module named 'BiochemPy'



## 5. Verify a few key reactions

In [6]:
key_rxns = {
    'rxn00148': 'Hexokinase (glucose + ATP -> G6P + ADP)',
    'rxn00199': 'Phosphofructokinase (F6P + ATP -> FBP + ADP)',
    'rxn00459': 'Pyruvate kinase (PEP + ADP -> Pyruvate + ATP)',
    'rxn00250': 'Citrate synthase (OAA + AcCoA -> Citrate + CoA)',
}

for rxn_id, name in key_rxns.items():
    if rxn_id in new_results:
        old = df_log[df_log['rxn_id'] == rxn_id].iloc[0] if rxn_id in df_log['rxn_id'].values else None
        new = new_results[rxn_id]
        print(f'{rxn_id} ({name}):')
        if old is not None:
            print(f'  Old dG: {old["old_deltag"]} kJ/mol')
        print(f'  New dG: {new["dG_mean"]:.2f} kJ/mol')
        print(f'  pH correction: {new["ddG0_pH_correction"]:.2f} kJ/mol')
    else:
        print(f'{rxn_id} ({name}): no prediction available')

rxn00148 (Hexokinase (glucose + ATP -> G6P + ADP)):
  Old dG: 6.53 kJ/mol
  New dG: 25.91 kJ/mol
  pH correction: -43.18 kJ/mol
rxn00199 (Phosphofructokinase (F6P + ATP -> FBP + ADP)):
  Old dG: -3.33 kJ/mol
  New dG: -18.02 kJ/mol
  pH correction: 44.86 kJ/mol
rxn00459 (Pyruvate kinase (PEP + ADP -> Pyruvate + ATP)):
  Old dG: -0.98 kJ/mol
  New dG: -4.27 kJ/mol
  pH correction: -0.00 kJ/mol
rxn00250 (Citrate synthase (OAA + AcCoA -> Citrate + CoA)):
  Old dG: -0.94 kJ/mol
  New dG: -5.34 kJ/mol
  pH correction: -41.58 kJ/mol
